<a href="https://colab.research.google.com/github/RubaEgbaria/RAG/blob/master/Howiyya_article_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # Web Scraper - howiyya.com Article Extraction
# Extracts full text from family stories and news articles using requests + BeautifulSoup.

# %% [code]
import re
import json
import time
import hashlib
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
import random

import requests
from bs4 import BeautifulSoup

# Configuration
OUTPUT_DIR = Path("scraped_data")
OUTPUT_DIR.mkdir(exist_ok=True)

CONFIG = {
    "min_words": 5,
    "delay": 5, # 5 sec delay
    "timeout": 30,
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "ar,en;q=0.9",
}

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Arabic-Indic digit normalization
def normalize_arabic_numerals(text: str) -> str:
    arabic_indic = '٠١٢٣٤٥٦٧٨٩'
    for i, ch in enumerate(arabic_indic):
        text = text.replace(ch, str(i))
    return text

# this is specific for extracting news from html parser
def extract_article_from_html(html: str) -> Dict:
    soup = BeautifulSoup(html, 'html.parser')

    for template in soup.find_all('template'):
        inner_soup = BeautifulSoup(template.decode_contents(), 'html.parser')
        template.replace_with(inner_soup)

    article_span = soup.select_one('span#el_vw_families_stories_Article')
    if not article_span:
        article_span = soup.select_one('span#el_vw_homenews_Article')

    if not article_span:
        return {"title": "", "text": "", "word_count": 0, "published_date": None}

    title_span = soup.select_one('span#el_vw_families_stories__Title')
    if not title_span:
        title_span = soup.select_one('span#el_vw_homenews__Title')
    title = title_span.get_text(strip=True) if title_span else ""

    raw_text = article_span.get_text(separator=' ', strip=True)
    clean_text = ' '.join(raw_text.split())

    published_date = None

    # Strategy 1: meta tag
    date_meta = soup.find("meta", attrs={"property": "article:published_time"})
    if date_meta and date_meta.get("content"):
        published_date = date_meta["content"]

    # Strategy 2: search the FULL page text, not just the article span
    # This catches "بتاريخ 17/12/2009" style footers that live outside the article span
    if not published_date:
        full_page_text = normalize_arabic_numerals(soup.get_text(separator=' ', strip=True))

        # Pattern: "بتاريخ DD/MM/YYYY" or "بتاريخ DD-MM-YYYY"
        date_match = re.search(
            r'بتاريخ\s+(\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4})',
            full_page_text
        )
        if date_match:
            published_date = date_match.group(1)

    # Strategy 3: fallback — any date-like pattern anywhere in full page
    if not published_date:
        full_page_text = normalize_arabic_numerals(soup.get_text(separator=' ', strip=True))
        date_match = re.search(r'(\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4})', full_page_text)
        if date_match:
            published_date = date_match.group(1)

    return {
        "title": title,
        "text": clean_text,
        "word_count": len(clean_text.split()),
        "published_date": published_date,
    }

# %% [markdown]
# ## Scraping Loop – Collect Multiple Articles

# %% [code]
def build_json_entry(url: str, extracted: Dict) -> Dict:
    from zoneinfo import ZoneInfo
    """Create a standardized JSON entry for an article."""
    page_id = hashlib.md5(url.encode()).hexdigest()[:8]
    return {
        "id": f"web_{page_id}",
        "ref": url,
        "ref_type": "article",
        "title": extracted["title"],
        "text": extracted["text"],
        "word_count": extracted["word_count"],
        "published_date": extracted.get("published_date"),
        "scraped_at": datetime.now(ZoneInfo("Asia/Jerusalem")).strftime("%Y-%m-%d %H:%M:%S"),
    }

def scrape_article(url: str) -> Optional[Dict]:
    """Fetch and extract a single article."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=CONFIG["timeout"])
        print(resp.url)
        if resp.status_code != 200:
            logger.warning(f"HTTP {resp.status_code} for {url}")
            return None
        extracted = extract_article_from_html(resp.text)
        if extracted["word_count"] < CONFIG["min_words"]:
            logger.info(f"Skipped {url} – too short ({extracted['word_count']} words)")
            return None
        return build_json_entry(url, extracted)
    except Exception as e:
        logger.error(f"Error scraping {url}: {e}")
        return None


# %% [markdown]
# ## Batch Scraping: Many IDs
# %% [code]
def scrape_range(start_id: int, end_id: int, page_type: str = "news"):
    """
    Scrape a range of article IDs.
    page_type: 'families' -> VwFamiliesStoriesView, 'news' -> VwNews5View / VwHomenewsView
    """
    results = []
    for article_id in range(start_id, end_id + 1):
        if page_type == "families":
            url = f"https://www.howiyya.com/VwFamiliesStoriesView?start={article_id}"
        else:
          url = f"https://www.howiyya.com/VwHomenewsView?start={article_id}"
            # url = f"https://www.howiyya.com/VwNews5View?start={article_id}"

        logger.info(f"Scraping {url}")
        article = scrape_article(url)
        if article:
            results.append(article)
            # Stream to JSONL
            with open(OUTPUT_DIR / f"{page_type}_stream.jsonl", "a", encoding="utf-8") as f:
                f.write(json.dumps(article, ensure_ascii=False) + "\n")

        time.sleep(CONFIG["delay"] + random.uniform(0.5, 2.0))


    # Save all results
    output_file = OUTPUT_DIR / f"{page_type}_{start_id}_{end_id}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    logger.info(f"Saved {len(results)} articles to {output_file}")
    return results



# Example: scrape a small range (adjust as needed)
results = scrape_range(2301, 2400, page_type="news")

https://www.howiyya.com/VwHomenewsView?start=2301
https://www.howiyya.com/VwHomenewsView?start=2302
https://www.howiyya.com/VwHomenewsView?start=2303
https://www.howiyya.com/VwHomenewsView?start=2304
https://www.howiyya.com/VwHomenewsView?start=2305
https://www.howiyya.com/VwHomenewsView?start=2306
https://www.howiyya.com/VwHomenewsView?start=2307
https://www.howiyya.com/VwHomenewsView?start=2308
https://www.howiyya.com/VwHomenewsView?start=2309
https://www.howiyya.com/VwHomenewsView?start=2310
https://www.howiyya.com/VwHomenewsView?start=2311
https://www.howiyya.com/VwHomenewsView?start=2312
https://www.howiyya.com/VwHomenewsView?start=2313
https://www.howiyya.com/VwHomenewsView?start=2314
https://www.howiyya.com/VwHomenewsView?start=2315
https://www.howiyya.com/VwHomenewsView?start=2316
https://www.howiyya.com/VwHomenewsView?start=2317
https://www.howiyya.com/VwHomenewsView?start=2318
https://www.howiyya.com/VwHomenewsView?start=2319
https://www.howiyya.com/VwHomenewsView?start=2320


In [ ]:
import statistics
import requests
import time

url = "https://www.howiyya.com/"
times = []

for _ in range(10):
    r = requests.head(url, timeout=10)
    times.append(r.elapsed.total_seconds())
    time.sleep(2)

print("avg:", statistics.mean(times))
print("max:", max(times))
print("min:", min(times))
print("stddev:", statistics.stdev(times))

avg: 0.5517548
max: 1.532344
min: 0.3722
stddev: 0.36859801090341704


In [ ]:
# avg: 0.5628317
# max: 0.674241
# min: 0.510635
# stddev: 0.045811520506551864

avg 255ms → healthy

min 125ms → very healthy

max 735ms → occasional slowdown, normal

stddev 195ms → some variance, but nothing alarming

In [ ]:
# take the text as input after run, remove the lines for json
text = input("Please enter your text: ")

text = text.replace("\n", " ")
text = text.replace('"', " ")

print(text)

Please enter your text: الأفواج الفلسطينية الأربعة هي:  فوج الشعراوية.  فوج صلاح الدين.  فوج الكرمل.  فوج خالد بن الوليد.
الأفواج الفلسطينية الأربعة هي:  فوج الشعراوية.  فوج صلاح الدين.  فوج الكرمل.  فوج خالد بن الوليد.
